# AML 3303 — Week 3 In-Class Lab (30 minutes)
### pandas + NumPy only

## Scenario

**NorthLine Electronics** is an online store. The ops team exported two tables for you:
`customers` and `orders`. They want a quick read on which product categories actually
make money, and which city orders the most.

The export is realistic, which means it is slightly messy: a missing quantity, a missing
price, a duplicate order, and one order from a customer who is not in the customer list.

Fill in every cell marked `# TODO`. The hints sit directly above each one.

In [3]:
print("Hello")

Hello


## 0. Setup

Run this cell once. Do not change it.

In [4]:
# Run this cell first. It creates the two DataFrames you will work with.
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Aarav Shah","Bianca Rossi","Chen Wei","Diya Patel",
             "Emeka Obi","Farah Haddad","Grace Kim","Hugo Martin"],
    "city": ["Ottawa","Toronto","Ottawa","Montreal",
             "Toronto","Calgary","Montreal","Ottawa"],
})

orders = pd.DataFrame({
    "order_id":   [101,102,103,104,105,106,107,108,109,110,
                   111,112,113,114,115,116,117,118,119,120,118,121],
    "customer_id":[  1,  2,  1,  3,  4,  5,  2,  6,  3,  7,
                     8,  5,  1,  4,  6,  7,  2,  8,  3,  5,  8,  9],
    "order_date": ["2024-05-02","2024-05-02","2024-05-04","2024-05-05","2024-05-07",
                   "2024-05-08","2024-05-09","2024-05-11","2024-05-12","2024-05-13",
                   "2024-05-15","2024-05-16","2024-05-18","2024-05-19","2024-05-20",
                   "2024-05-22","2024-05-23","2024-05-25","2024-05-26","2024-05-28",
                   "2024-05-25","2024-05-29"],
    "category":   ["Laptops","Headphones","Headphones","Monitors","Laptops",
                   "Keyboards","Monitors","Laptops","Keyboards","Headphones",
                   "Monitors","Laptops","Keyboards","Headphones","Monitors",
                   "Laptops","Keyboards","Headphones","Laptops","Monitors",
                   "Headphones","Keyboards"],
    "quantity":   [1,2,1,2,1,3,1,1,np.nan,1,4,1,2,1,1,1,1,2,1,2,2,1],
    "unit_price": [1299.99,89.50,89.50,219.00,1499.00,45.00,219.00,1099.00,45.00,np.nan,
                   219.00,1299.99,45.00,89.50,219.00,1499.00,45.00,89.50,1299.99,219.00,
                   89.50,45.00],
    "status":     ["delivered","delivered","returned","delivered","cancelled",
                   "delivered","delivered","delivered","delivered","delivered",
                   "delivered","returned","delivered","delivered","delivered",
                   "delivered","cancelled","delivered","delivered","delivered",
                   "delivered","delivered"],
})

print("orders:", orders.shape, " customers:", customers.shape)
orders.head(22)

orders: (22, 7)  customers: (8, 3)


,order_id,customer_id,order_date,category,quantity,unit_price,status
0,101,1,2024-05-02,Laptops,1.0,1299.99,delivered
1,102,2,2024-05-02,Headphones,2.0,89.50,delivered
2,103,1,2024-05-04,Headphones,1.0,89.50,returned
3,104,3,2024-05-05,Monitors,2.0,219.00,delivered
4,105,4,2024-05-07,Laptops,1.0,1499.00,cancelled
5,106,5,2024-05-08,Keyboards,3.0,45.00,delivered
6,107,2,2024-05-09,Monitors,1.0,219.00,delivered
7,108,6,2024-05-11,Laptops,1.0,1099.00,delivered
8,109,3,2024-05-12,Keyboards,NaN,45.00,delivered
9,110,7,2024-05-13,Headphones,1.0,NaN,delivered


## 1. First look (3 min)

**Hints**
- `df.shape` → rows and columns
- `df.info()` → column types and non-null counts
- `df.describe()` → quick numeric summary
- `df["category"].value_counts()` → how many orders per category

In [5]:
# TODO: print the shape, run info(), and show how many orders there are per category
print(orders.shape)
print(orders.info())
print(orders.describe())
orders["category"].value_counts()

(22, 7)
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     22 non-null     int64  
 1   customer_id  22 non-null     int64  
 2   order_date   22 non-null     str    
 3   category     22 non-null     str    
 4   quantity     21 non-null     float64
 5   unit_price   21 non-null     float64
 6   status       22 non-null     str    
dtypes: float64(2), int64(2), str(3)
memory usage: 1.3 KB
None
         order_id  customer_id   quantity   unit_price
count   22.000000    22.000000  21.000000    21.000000
mean   111.318182     4.545455   1.523810   464.974762
std      6.236501     2.558409   0.813575   571.028639
min    101.000000     1.000000   1.000000    45.000000
25%    106.250000     2.250000   1.000000    89.500000
50%    111.500000     4.500000   1.000000   219.000000
75%    116.750000     6.750000   2.000000  1099.000000
max    121.000000   

category
Laptops       6
Headphones    6
Monitors      5
Keyboards     5
Name: count, dtype: int64

## 2. A first (wrong) answer (5 min)

Compute revenue per category **before** cleaning anything, counting delivered orders only.
Keep this result: you will compare against it at the end.

**Hints**
- Filter rows: `orders[orders["status"] == "delivered"]`
- New column: `df["revenue"] = df["quantity"] * df["unit_price"]`
- Group: `df.groupby("category")["revenue"].sum().sort_values(ascending=False)`

In [6]:
# TODO: revenue by category using the RAW orders (delivered only)
revenue_raw = orders[orders["status"] == "delivered"]
revenue_raw["revenue"] = revenue_raw["quantity"] * revenue_raw["unit_price"]
revenue_raw["revenue"]
revenue_raw = revenue_raw.groupby("category")["revenue"].sum().sort_values(ascending=False)
revenue_raw

category
Laptops       5197.98
Monitors      2190.00
Headphones     626.50
Keyboards      270.00
Name: revenue, dtype: float64

## 3. Find the problems (5 min)

**Hints**
- Duplicates: `df.duplicated().sum()`, and `df[df.duplicated(keep=False)]` to see them
- Missing values: `df.isna().sum()`
- Orphan rows: `~orders["customer_id"].isin(customers["customer_id"])`

In [12]:
# TODO: 1. how many fully duplicated rows are there? Show them.

df = orders
df.duplicated().sum()
duplications = df[df.duplicated(keep=False)]
display(duplications)
# df

# TODO: 2. how many missing values in each column?
print(df.isna().sum())

# TODO: 3. which orders belong to a customer who is NOT in the customers table?
orders_not_in_customers = orders.merge(
    customers[["customer_id"]].drop_duplicates(),
    on="customer_id",
    how="left",
    indicator=True
)

orders_not_in_customers = orders_not_in_customers[
    orders_not_in_customers["_merge"] == "left_only"
].drop(columns="_merge")

orders_not_in_customers

,order_id,customer_id,order_date,category,quantity,unit_price,status
17,118,8,2024-05-25,Headphones,2.0,89.5,delivered
20,118,8,2024-05-25,Headphones,2.0,89.5,delivered


order_id       0
customer_id    0
order_date     0
category       0
quantity       1
unit_price     1
status         0
dtype: int64


,order_id,customer_id,order_date,category,quantity,unit_price,status
21,121,9,2024-05-29,Keyboards,1.0,45.0,delivered


## 4. Clean it (7 min)

**Hints**
- `orders_clean = orders.drop_duplicates()`
- Fill a price with the median for that category:
  `df.groupby("category")["unit_price"].transform("median")`
- `df["quantity"].fillna(1)` — one unit is a reasonable default for this shop
- `pd.to_datetime(df["order_date"])`
- Make quantity an integer at the end: `.astype(int)`

In [13]:
orders_clean = orders.copy()

# TODO: 1. drop duplicate rows
orders_clean = orders_clean.drop_duplicates()

# TODO: 2. fill missing unit_price with the median price of the same category
category_medians = orders_clean.groupby("category")["unit_price"].transform("median")
orders_clean["unit_price"] = orders_clean["unit_price"].fillna(category_medians)

# TODO: 3. fill missing quantity with 1, then convert the column to int
orders_clean["quantity"] = orders_clean["quantity"].fillna(1).astype(int)


# TODO: 4. convert order_date to datetime
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])

orders_clean.info()

<class 'pandas.DataFrame'>
Index: 21 entries, 0 to 21
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     21 non-null     int64         
 1   customer_id  21 non-null     int64         
 2   order_date   21 non-null     datetime64[us]
 3   category     21 non-null     str           
 4   quantity     21 non-null     int64         
 5   unit_price   21 non-null     float64       
 6   status       21 non-null     str           
dtypes: datetime64[us](1), float64(1), int64(3), str(2)
memory usage: 1.3 KB


## 5. NumPy: revenue column (3 min)

Cancelled and returned orders earn **no** revenue.

**Hints**
- `np.where(condition, value_if_true, value_if_false)`
- Condition: `orders_clean["status"] == "delivered"`
- `np.round(x, 2)` if you want tidy numbers

In [14]:
# TODO: create a "revenue" column that is 0 for cancelled and returned orders
orders_clean["revenue"] = np.where(
    orders_clean["status"] == "delivered",
    orders_clean["quantity"] * orders_clean["unit_price"],
    0
)

orders_clean["revenue"] = np.round(orders_clean["revenue"], 2)


# TODO: print total revenue, the mean order value, and the largest single order
# Summary statistics
print("Total revenue:", round(orders_clean["revenue"].sum(), 2))
print("Mean order value:", round(orders_clean["revenue"].mean(), 2))
print("Largest single order:", round(orders_clean["revenue"].max(), 2))

Total revenue: 8239.98
Mean order value: 392.38
Largest single order: 1499.0


## 6. Merge and answer the business question (5 min)

**Hints**
- `orders_clean.merge(customers, on="customer_id", how="left")`
- A left merge keeps the order from the unknown customer, with `city` as NaN. That is
  usually what you want: do not silently delete money.
- `.groupby("city")["revenue"].sum().sort_values(ascending=False)`

In [15]:
# TODO: merge customers into orders_clean, then find revenue by city
merged = orders_clean.merge(
    customers[["customer_id", "city"]],
    on="customer_id",
    how="left"
)

# Calculate total revenue by city
revenue_by_city = (
    merged.groupby("city", as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
)

revenue_by_city

,city,revenue
2,Ottawa,4227.98
1,Montreal,1678.00
0,Calgary,1318.00
3,Toronto,971.00


## 7. Compare with step 2 (2 min)

**Hint:** `pd.DataFrame({"raw": revenue_raw, "clean": revenue_clean})` lines the two up side by side.

In [21]:
# TODO: revenue by category from the CLEANED data, then compare it with revenue_raw
evenue_clean = orders_clean.groupby("category")["revenue"].sum()

# Compare raw and cleaned revenue side by side
comparison = pd.DataFrame({
    "raw": revenue_raw,
    "clean": revenue_clean
})

comparison

,raw,clean
category,,
Laptops,5197.98,Ellipsis
Monitors,2190.00,Ellipsis
Headphones,626.50,Ellipsis
Keyboards,270.00,Ellipsis


**Question:** two categories changed. Which ones, and what caused each change?

*Your answer:*
Push Laptops: they bring in $5,197.98 of the $8,239.98 total delivered revenue (about 63%), far from Monitors at $2,190. At the source, apply a unique order_id and required quantity and unit_price fields in the export, because a duplicated order (118) and missing values were silently distorting the revenue numbers.



## 8. Wrap up (optional, if time)

Write two sentences for the NorthLine ops team: which category to push, and one data
quality issue they should fix at the source.

*Headphones decrease from $626.50 to $537.00. Removing the duplicate order 118 took out $179. Filling the missing price of order 110 with the category median $89.50 added $89.50 that the raw version had taked as NaN. Keyboards went up from $270.00 to $315.00. Order 109 had a missing quantity, so the raw revenue was NaN and was ignored. Filling it with 1 added 1 × $45 = $45. Laptops and Monitors did not change because none of their delivered orders had problems*

---

### Submission
Save the notebook, push it to your GitHub repo, and paste the repo link in the class channel.

### Stretch goals
1. Which customer spent the most? (merge, then group by name)
2. Add a `week` column from `order_date` and show revenue per week.
   Hint: `df["order_date"].dt.isocalendar().week`
3. Ask Claude or Copilot to write the cleaning step for you, then compare it with yours.
   Did it make the same choices about the missing quantity? Was it right?